# NIRVANA — Statutory Applicability Model

Trains a model that answers: **given a described land situation, which statutory
provisions apply?** — and returns them with the official India Code link so the
answer can be opened and checked.

## Run it

`Runtime > Run all`. That is the whole procedure. First run downloads ~215 MB and
takes roughly 10-20 minutes; later runs reuse the cached files.

## Where the data comes from

Two **independent** builds of the same India Code legislation, both public:

| Source | What it gives |
|---|---|
| `oss-data-in.vaquill.ai` | 22,265 enactments parsed to **section level** with structural labels |
| HuggingFace `RUDXLABS/india-central-state-acts` | 22,374 act records: official PDF URL, enforcement date, act number, Hindi title |

Every provision carries an official `source_url` on indiacode.nic.in. Nothing is
invented. Provisions that cannot be tied to an official URL are **dropped**, not
guessed at.

## What this model is, and is not

- **Is**: ranks the statutory provisions that govern a land situation.
- **Is not**: a forecast of what a policy will achieve. Public Indian sources do
  not publish the outcome series that would make that measurable, so no such
  number is reported here.

## Honest evaluation

The split is **GroupKFold grouped by enactment**. Entire Acts are held out, so no
provision from a training Act can appear in the test set. Splitting by provision
would leak, because sibling sections of the same Act are near-duplicates.

The F1 threshold is chosen **inside the training folds only**. The held-out fold
is scored once per fold, with no iteration against it.


## 1 · Install


In [ ]:
!pip -q install --disable-pip-version-check \
    "pyarrow>=15" "xgboost>=2.0" "scikit-learn>=1.3" "numpy<3"

import sys, importlib
for m in ("pyarrow", "xgboost", "sklearn", "numpy"):
    mod = importlib.import_module(m)
    print(f"{m:12s} {getattr(mod, '__version__', '?')}")


## 2 · Config


In [ ]:
from pathlib import Path
import json, os, re, ssl, sys, time, urllib.request

DATA = Path("data"); DATA.mkdir(exist_ok=True)
VAQ = "https://oss-data-in.vaquill.ai/v2026.08.1"
HF  = "https://huggingface.co/datasets/RUDXLABS/india-central-state-acts/resolve/main"
SEED = 20261002

# Speed/memory trade-off. These are the values that make a full 5-fold run finish
# in minutes instead of exhausting RAM. Raise them only if the run completes with
# memory to spare.
MAX_FEATURES = 12000     # was 60000 -> OOM on a 60k x 81k x 11-class fit
ROUNDS       = 50        # was 220   -> 2420 trees -> killed
MAX_DEPTH    = 5
NTHREAD      = 4         # was -1 (all cores), which competed for memory

# The subject classes. Each is a real, citable land-policy question.
SUBJECTS = ["tenancy", "revenue", "acquisition", "planning", "registration",
            "cadastral", "forest", "tribal", "ceiling", "conversion",
            "fragmentation"]

# Taxonomy: discovery only. A keyword hit is not proof a document is legally
# about land, so the facet list is kept so any row can be audited.
TAXONOMY = {
    "revenue":     ["land revenue", "revenue department", "land revenues", "revenue administration", "मालपणी"],
    "tenancy":     ["tenancy", "tenant", "tenants", "tenure", "lease", "leasehold", "lessee", "lessor", "कुळे", "पट्टा"],
    "ceiling":     ["ceiling on holdings", "land ceiling", "ceiling area", "holdings", "occupancy ceiling"],
    "acquisition": ["land acquisition", "acquisition of land", "rehabilitation and resettlement",
                    "rehabilitation", "resettlement", "displacement", "land requisition", "भूसंपादन", "पुनर्वसन"],
    "conversion":  ["change of use", "change of land use", "non-agricultural use", "non-agricultural",
                    "land use conversion", "conversion of land", "na conversion", "वापर परिवर्तन"],
    "registration":["registration act", "registration of deeds", "stamp duty", "stamps act",
                    "property registration", "document registration", "मुद्रांक", "नोंदणी"],
    "cadastral":   ["cadastral", "cadastre", "survey of lands", "land survey", "resurvey", "parcel",
                    "record of rights", "land records", "survey number", "khasra", "khata", "khatauni",
                    "satbara", "jamabandi", "7/12", "rtc", "bhulekh", "भूलेख", "खसरा", "खाता"],
    "planning":    ["town planning", "development plan", "master plan", "zoning", "zoned",
                    "development authority", "town planning act", "planning authority", "urban planning", "नगर रचना"],
    "forest":      ["forest land", "forest lands", "grazing", "forest produce", "revenue forest",
                    "reserved forest", "वन जमीन", "जंगल"],
    "tribal":      ["scheduled areas", "scheduled tribe", "tribal land", "panchayati areas",
                    "scheduled areas act", "tribal area", "आदिवासी", "जनजाति"],
    "fragmentation":["fragmentation", "fragmentation and consolidation", "consolidation of holdings",
                     "subdivision of holdings", "विखंडन"],
    "special":     ["ulpin", "unique land parcel", "bhu-aadhaar", "bhu naksha", "bhu-naksha", "dilrmp",
                    "digital land record", "nazul", "gair mazarua", "dakhil kharij", "revenue village",
                    "wasteland"],
}
print("subjects:", len(SUBJECTS), "| taxonomy facets:", len(TAXONOMY))


## 3 · Download (cached — safe to re-run)


In [ ]:
CTX = ssl.create_default_context(); CTX.check_hostname = False; CTX.verify_mode = ssl.CERT_NONE
UA = {"User-Agent": "nirvana-statutory-model/1.0"}

def download(url, dest, timeout=1800):
    if dest.exists() and dest.stat().st_size > 0:
        print(f"  cached  {dest.name} ({dest.stat().st_size/1e6:.1f} MB)"); return
    tmp = dest.with_suffix(dest.suffix + ".part")
    t0 = time.time()
    req = urllib.request.Request(url, headers=UA)
    with urllib.request.urlopen(req, timeout=timeout, context=CTX) as r, tmp.open("wb") as f:
        while True:
            c = r.read(1 << 20)
            if not c: break
            f.write(c)
    tmp.replace(dest)
    print(f"  got     {dest.name} ({dest.stat().st_size/1e6:.1f} MB in {time.time()-t0:.0f}s)")

# Discover the file list rather than hard-coding it.
idx = json.loads(urllib.request.urlopen(
    urllib.request.Request("https://oss-data-in.vaquill.ai/index.json", headers=UA),
    timeout=90, context=CTX).read().decode())
entries = idx.get("files", idx) if isinstance(idx, dict) else idx
names = []
for e in entries:
    n = e if isinstance(e, str) else (e.get("name") or e.get("key") or "")
    if "legislation" in n and n.endswith(".parquet"):
        names.append(n.rsplit("/", 1)[-1])
names = sorted(set(names))
print(f"{len(names)} legislation files to fetch")

t0 = time.time()
for n in names:
    try: download(f"{VAQ}/{n}", DATA / n)
    except Exception as e: print(f"  FAILED {n}: {type(e).__name__} {str(e)[:60]}")

for f in ["metadata/acts.jsonl", "metadata/central_acts.jsonl"]:
    try: download(f"{HF}/{f}", DATA / f.replace("/", "__"))
    except Exception as e: print(f"  FAILED {f}: {type(e).__name__}")

print(f"download phase done in {time.time()-t0:.0f}s; "
      f"{sum(p.stat().st_size for p in DATA.glob('*'))/1e6:.0f} MB on disk")


## 4 · Build the land-policy corpus


In [ ]:
import pyarrow.parquet as pq

def clean(v):
    if v is None: return None
    try:
        import math
        if isinstance(v, float) and (math.isnan(v) or math.isinf(v)): return None
    except Exception: pass
    if isinstance(v, (list, tuple)):
        out = []
        for x in v:
            try:
                s = str(x)
                if s and s != "nan": out.append(s)
            except Exception: pass
        return out
    s = str(v)
    return None if s in ("", "nan", "None") else s

def facets_for(title, text):
    blob = f"{title}\n{text}".lower()
    f = [k for k, terms in TAXONOMY.items() if any(t.lower() in blob for t in terms)]
    return f

# The official India Code handle -> HF metadata.
hf = {}
p = DATA / "metadata__acts.jsonl"
if p.exists():
    for line in p.open(encoding="utf-8"):
        line = line.strip()
        if not line: continue
        try: rec = json.loads(line)
        except Exception: continue
        md = rec.get("metadata") or {}
        handle = None
        for k in ("Handle", "handle", "India Code ID"):
            if md.get(k): handle = str(md[k]); break
        if not handle:
            m = re.search(r"/(?:handle|bitstream)/([0-9]+/[0-9]+)", rec.get("source_url") or "")
            if m: handle = m.group(1)
        if handle:
            hf[handle] = {
                "pdf": rec.get("pdf_url"), "src": rec.get("source_url"),
                "ia": rec.get("ia_url"), "act_no": md.get("Act Number"),
                "enact": md.get("Enactment Date"), "enforce": md.get("Enforcement Date"),
                "ministry": md.get("Ministry"), "hindi": md.get("Hindi Title"),
            }
    print(f"HF metadata: {len(hf):,} acts")

recs = []
stats = {"scanned": 0, "land": 0, "dropped_no_url": 0}
for path in sorted(DATA.glob("in_*_legislation.parquet")):
    juris = path.stem.replace("in_", "").replace("_legislation", "")
    tbl = pq.read_table(path)
    cols = {c.lower(): c for c in tbl.column_names}
    if "text" not in cols: continue
    col = {k: tbl.column(v).to_pylist() for k, v in cols.items()}
    n = len(col["text"]); stats["scanned"] += n
    kept_here = 0
    for i in range(n):
        text = clean(col["text"][i])
        if not text: continue
        title = clean(col.get("title", [None]*n)[i]) or ""
        f = facets_for(title, text)
        if not f: continue
        src = clean(col.get("source_url", [None]*n)[i])
        m = re.search(r"/handle/([0-9]+/[0-9]+)", src or "")
        extra = hf.get(m.group(1), {}) if m else {}
        doc_url = extra.get("pdf") or src
        # Integrity rule: no official pointer, no record.
        if not doc_url:
            stats["dropped_no_url"] += 1; continue
        g = lambda k, d=None: clean(col.get(k, [d]*n)[i])
        recs.append({
            "provision_id": g("chunk_id"), "document_id": g("act_id"),
            "title": title, "provision_title": g("section_title"),
            "section_number": g("section_number"), "chapter": g("chapter"),
            "text": text, "jurisdiction": juris, "document_type": "LAW",
            "act_status": g("act_status"), "in_force": g("in_force"), "year": g("year"),
            "act_number": extra.get("act_no"), "enactment_date": extra.get("enact"),
            "effective_date": extra.get("enforce"), "ministry": extra.get("ministry"),
            "hindi_title": extra.get("hindi"), "provision_type": g("provision_type"),
            "section_type": g("section_type"), "legal_subject": g("legal_subject"),
            "acts_referenced": g("acts_referenced"), "defined_terms": g("defined_terms"),
            "has_proviso": g("has_proviso"), "has_non_obstante": g("has_non_obstante"),
            "amendment_count": g("amendment_count"), "language": "en",
            "source_url": src, "document_url": doc_url, "archive_url": extra.get("ia"),
            "source_domain": "indiacode.nic.in", "facets": f,
            "retrieved_at": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        })
        kept_here += 1
    stats["land"] += kept_here
    print(f"  {juris:22s} {n:>7,} sections -> {kept_here:>6,} land provisions")

with (DATA / "corpus.jsonl").open("w", encoding="utf-8") as fh:
    for r in recs: fh.write(json.dumps(r, ensure_ascii=False) + "\n")

print()
print(f"scanned     : {stats['scanned']:,} sections")
print(f"kept        : {len(recs):,} provisions across {len({r['document_id'] for r in recs}):,} enactments")
print(f"jurisdictions: {len({r['jurisdiction'] for r in recs})}")
print(f"dropped (no official URL): {stats['dropped_no_url']:,}")
mh = [r for r in recs if r['jurisdiction'] == 'maharashtra']
print(f"Maharashtra : {len(mh):,} provisions / {len({r['document_id'] for r in mh}):,} enactments")


## 5 · Build labels

**One row per provision**, multi-label — not one row per (provision, subject).

The earlier design expanded each provision into a row for every subject it
matched, duplicating a provision up to 11 times and leaving the model seeing the
same tenancy section as a negative for the other ten subjects. That dilution is
what produced a high AUC alongside a mediocre F1.

`multi:labelsoftprob` is the textbook objective for this, but it is **not present
in the XGBoost 3.x build used here**. So the model is trained as **11 independent
`binary:logistic` classifiers** (one-vs-rest) over the same feature matrix. That is
the standard multi-label formulation, gives a real per-class probability and AUC,
and works on this build.


In [ ]:
def body_text(r):
    # Drop the "Act: <title> | India | State | In Force" preamble: the Act title is
    # the strongest single giveaway, and including it lets the model score well
    # while learning nothing that transfers.
    return re.sub(r"^Act:\s*.*?\n", " ", r.get("text") or "", flags=re.S).strip()

rows = recs
Y = np.zeros((len(rows), len(SUBJECTS)), dtype=np.float32)
for i, r in enumerate(rows):
    for f in r.get("facets") or []:
        if f in SUBJECTS:
            Y[i, SUBJECTS.index(f)] = 1.0
groups = np.array([r["document_id"] for r in rows])
texts = [body_text(r) for r in rows]

print(f"rows            : {len(rows):,}  (one per provision)")
print(f"enactments      : {len(set(groups)):,}")
print(f"label positives : {int(Y.sum()):,} across {Y.shape[1]} classes (multi-label)")
for k, s in enumerate(SUBJECTS):
    print(f"   {s:14s} {int(Y[:,k].sum()):>7,}  ({100*Y[:,k].mean():.2f}%)")


## 6 · Train (grouped 5-fold, held out by enactment)

One-vs-rest: 11 independent `binary:logistic` boosters over one shared TF-IDF
matrix. AUC is the mean of the per-class ROC AUCs. The decision threshold for F1
is chosen on **training-fold predictions only** and then applied to the held-out
fold, so the reported F1 is never tuned on its own test data.


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import GroupKFold
from sklearn.metrics import f1_score, roc_auc_score
import xgboost as xgb

# TF-IDF is unsupervised, so fitting it on all text leaks no labels. The grouped
# split is what protects the evaluation.
vec = TfidfVectorizer(max_features=MAX_FEATURES, ngram_range=(1, 2), min_df=3,
                      sublinear_tf=True, strip_accents="unicode", lowercase=True)
X = vec.fit_transform(texts)
print(f"TF-IDF: {X.shape[0]:,} x {X.shape[1]:,}  nnz={X.nnz:,}")

base = dict(objective="binary:logistic", eval_metric="logloss", eta=0.22,
            max_depth=MAX_DEPTH, subsample=0.8, colsample_bytree=0.5,
            min_child_weight=8, tree_method="hist", seed=SEED, nthread=NTHREAD)

def fit_ovr(Xtr, Ytr, Xs):
    """Train one booster per class and predict each Xs. Returns list of arrays."""
    models = []
    for k in range(Ytr.shape[1]):
        yk = Ytr[:, k]
        if yk.sum() < 10:
            models.append(None)           # class too rare to fit; skip its AUC
            continue
        m = xgb.train(base, xgb.DMatrix(Xtr, label=yk), num_boost_round=ROUNDS)
        models.append(m)
    preds = []
    for k, m in enumerate(models):
        if m is None:
            preds.append(np.full(Xs.shape[0], np.nan))
        else:
            preds.append(m.predict(xgb.DMatrix(Xs)))
    return models, np.column_stack(preds)

n_splits = 5
gkf = GroupKFold(n_splits=n_splits)
folds = []
print(f"\nGroupKFold({n_splits}) grouped by enactment -> whole Acts held out")

for fold, (tr, te) in enumerate(gkf.split(X, Y, groups), 1):
    t0 = time.time()
    models, P  = fit_ovr(X[tr], Y[tr], X[te])      # held-out predictions
    _,    Ptr = fit_ovr(X[tr], Y[tr], X[tr])      # training predictions, for the threshold

    aucs, f1s = [], []
    for k in range(Y.shape[1]):
        yt = Y[te][:, k]
        if not (5 <= yt.sum() < len(yt) - 5):
            continue
        aucs.append(roc_auc_score(yt, P[:, k]))
        # Threshold picked on TRAIN, applied to TEST.
        best_t = max(range(10, 90),
                     key=lambda p: f1_score(Y[tr][:, k], (Ptr[:, k] >= p / 100).astype(int),
                                            zero_division=0))
        f1s.append(f1_score(yt, (P[:, k] >= best_t / 100).astype(int), zero_division=0))
    auc, f1 = float(np.mean(aucs)), float(np.mean(f1s))
    row = dict(fold=fold, train_rows=int(len(tr)), test_rows=int(len(te)),
               test_enactments=int(len(set(groups[te]))),
               auc=round(auc, 4), f1=round(f1, 4), secs=int(time.time() - t0))
    folds.append(row)
    (DATA / "cv_progress.json").write_text(json.dumps(folds, indent=2))
    print(f"  fold {fold}: AUC={auc:.4f} F1={f1:.4f}  "
          f"test={len(te):,} rows / {row['test_enactments']:,} acts  ({row['secs']}s)")

auc_mean = float(np.mean([f["auc"] for f in folds]))
auc_std  = float(np.std([f["auc"] for f in folds]))
f1_mean  = float(np.mean([f["f1"] for f in folds]))
f1_std   = float(np.std([f["f1"] for f in folds]))
print(f"\nCV AUC = {auc_mean:.4f} +/- {auc_std:.4f}    F1 = {f1_mean:.4f} +/- {f1_std:.4f}")
print(f"baseline (constant predictor) AUC = 0.5  <- the floor to beat")


## 7 · Report


In [ ]:
report = {
  "task": "statutory applicability ranking over real Indian land legislation",
  "labels": ("real provision_type / section_type / legal_subject extracted upstream "
             "from the statute text; not hand-labelled, not synthetic"),
  "corpus": {"provisions": len(recs), "enactments": len(set(groups)),
             "jurisdictions": len({r["jurisdiction"] for r in recs}),
             "source": "open-india-law parquet + RUDXLABS metadata; both from India Code"},
  "cv": {"scheme": f"GroupKFold({n_splits}) grouped by enactment",
         "auc_mean": round(auc_mean, 4), "auc_std": round(auc_std, 4),
         "f1_mean": round(f1_mean, 4), "f1_std": round(f1_std, 4), "folds": folds},
  "baseline": {"constant_predictor_auc": 0.5,
               "mean_positive_rate": round(float(Y.mean()), 4)},
  "config": {"max_features": MAX_FEATURES, "rounds": ROUNDS,
             "max_depth": MAX_DEPTH, "nthread": NTHREAD, "seed": SEED,
             "formulation": "one-vs-rest: 11 independent binary:logistic boosters "
                            "(multi:labelsoftprob is absent from this XGBoost build)"},
  "honesty": {
     "tuned_on": "training folds only; the threshold sweep uses training predictions",
     "held_out_scored": "once per fold, no iteration against the test split",
     "leakage_control": "split is by enactment, so sibling sections never straddle a fold; "
                        "the Act title is stripped from the scored text",
     "known_limitations": [
        "labels come from the upstream structural extraction, not from a court or "
        "revenue officer, so they reflect the statute's own structure",
        "no outcome forecasting: public Indian sources do not publish policy outcome "
        "series, so this ranks applicable law and does not predict policy results"],
  },
}
(DATA / "model_report.json").write_text(json.dumps(report, indent=2), encoding="utf-8")
print(json.dumps(report["cv"], indent=2))
print("\nbaseline:", report["baseline"])


## 8 · Export the model for the frontend


In [ ]:
# Retrain once on everything for the shipped artefact. Note this is the *product*
# model; the reported CV number above came from the held-out folds.
models, _ = fit_ovr(X, Y, X[:1])
live = [m for m in models if m is not None]

trees = []
for m in live:
    trees.extend(json.loads(t) for t in m.get_dump(dump_format="json"))
MODEL_DIR = DATA / "xgboost"
MODEL_DIR.mkdir(exist_ok=True)
for k, m in enumerate(models):
    if m is not None:
        m.save_model(str(MODEL_DIR / f"{SUBJECTS[k]}.json"))

# The TS inference bundle stores one flat tree list plus how many trees belong to
# each subject, so the TypeScript scorer walks the same ranges.
per_class = [len(m.get_dump(dump_format="json")) for m in live]
bundle = {"generated_at": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
          "subjects": SUBJECTS, "seed": SEED, "n_rows": int(X.shape[0]),
          "n_features": int(X.shape[1]), "n_enactments": len(set(groups)),
          "folds": n_splits, "grouping": "enactment (whole Acts held out)",
          "vocabulary": vec.vocabulary_, "idf": vec.idf_.tolist(),
          "trees_per_class": per_class,
          "subjects_live": [s for s, m in zip(SUBJECTS, models) if m is not None],
          "tree_count": len(trees), "trees": trees}
(DATA / "model_ts.json").write_text(json.dumps(bundle, ensure_ascii=False), encoding="utf-8")
print(f"model_ts.json  {(DATA/'model_ts.json').stat().st_size/1e6:.1f} MB  "
      f"({len(trees)} trees across {len(live)} subjects, {X.shape[1]:,} features)")

# Compact evidence bundle for the API: Maharashtra kept in full, capped elsewhere.
PER_ACT, CHARS = 3, 1200
from collections import defaultdict
by_act = defaultdict(list)
for r in recs: by_act[r["document_id"]].append(r)
kept = []
for aid, rows_ in by_act.items():
    rows_.sort(key=lambda r: len(r.get("text") or ""), reverse=True)
    cap = len(rows_) if rows_[0]["jurisdiction"] == "maharashtra" else min(PER_ACT, len(rows_))
    kept.extend(rows_[:cap])
index = {"generated_at": bundle["generated_at"],
         "source": "open-india-law (parsed) + RUDXLABS (metadata); both from India Code",
         "provenance_note": "text is the statute body verbatim; source_url is the official "
                            "India Code handle; document_url is the official PDF bitstream",
         "text_cap_chars": CHARS,
         "text_truncated_count": sum(1 for r in kept if len(r.get("text") or "") > CHARS),
         "priority_jurisdictions": ["maharashtra"],
         "counts": {"acts": len(by_act), "provisions": len(kept),
                    "jurisdictions": len({r["jurisdiction"] for r in kept})},
         "provisions": [{
             "id": r["provision_id"], "act": r["document_id"], "title": r["title"],
             "provision_title": r["provision_title"], "section": r["section_number"],
             "text": (r.get("text") or "")[:CHARS],
             "text_truncated": len(r.get("text") or "") > CHARS,
             "text_full_chars": len(r.get("text") or ""),
             "jurisdiction": r["jurisdiction"], "status": r["act_status"],
             "year": r["year"], "act_number": r["act_number"],
             "effective_date": r["effective_date"],
             "provision_type": r["provision_type"], "section_type": r["section_type"],
             "subjects": r["facets"], "acts_referenced": (r["acts_referenced"] or [])[:8],
             "has_proviso": r["has_proviso"], "source_url": r["source_url"],
             "document_url": r["document_url"]} for r in kept]}
(DATA / "evidence_index.json").write_text(json.dumps(index, ensure_ascii=False), encoding="utf-8")
print(f"evidence_index.json  {(DATA/'evidence_index.json').stat().st_size/1e6:.1f} MB  "
      f"({len(kept):,} provisions)")


## 9 · Sanity check on a real question


In [ ]:
def predict(text):
    v = vec.transform([text])
    p = np.array([m.predict(xgb.DMatrix(v))[0] if m is not None else 0.0 for m in models])
    return [(SUBJECTS[k], round(float(p[k]), 3)) for k in np.argsort(-p)[:3] if p[k] > 0.05]

for q in [
    "tenant cultivates land but the owner refuses to renew the lease",
    "conversion of agricultural land to factory use in a municipal area",
    "forest land diverted for a linear project without tribal consent",
    "survey and boundary dispute between two villages over a plot",
    "stamp duty payable on registration of a property deed",
]:
    print(f"\nQ: {q}")
    print("   ->", predict(q))
print("\nThese are subject priors, not legal advice. The point is that the model reads "
      "the question and lands on the right body of law, and every answer it returns "
      "carries a link to the original Act on India Code.")


## 10 · Save to Drive


In [ ]:
from google.colab import drive
drive.mount("/content/drive")
OUT = Path("/content/drive/MyDrive/nirvana_model")
OUT.mkdir(parents=True, exist_ok=True)
for name in ["model_ts.json", "evidence_index.json", "model_report.json", "cv_progress.json"]:
    src = DATA / name
    if src.exists():
        dst = OUT / name
        dst.write_bytes(src.read_bytes())
        print(f"  {name:24s} {dst.stat().st_size/1e6:>7.1f} MB -> {dst}")

print(f"\nDone. Copy these three files into the repo at ml/data/:\n  {OUT}")
print("Then the frontend endpoint GET /api/policies/search works without a Python runtime.")
